# MATRES DATA EXPLORATION

Using a pre-processed MATRES dataset from NeuralTemporalRelation-EMNLP19


In [1]:
import xml.etree.ElementTree as ET # module for parsing xml data

In [2]:
train_path = "data/NeuralTemporalRelation-EMNLP19/data/trainset-temprel.xml"
tree = ET.parse(train_path)
root = tree.getroot()
print(root.tag)
print(len(root))

DATA
12740


In [3]:
example = root[0]
print(example.tag)
print(example.attrib)
print(example.text[:100])

SENTENCE
{'DOCID': 'ABC19980120.1830.0957', 'SOURCE': 'E415', 'TARGET': 'E417', 'SOURCE_SENTID': '1', 'TARGET_SENTID': '1', 'LABEL': 'BEFORE', 'SENTDIFF': '0'}
People///people///NNS///B have///have///VBP///B predicted///predict///VBN///E1 his///his///PRP$///M 


In [4]:
tokens = example.text.strip().split()
print(tokens[:5])
#word ///lemma///POS///marker

['People///people///NNS///B', 'have///have///VBP///B', 'predicted///predict///VBN///E1', 'his///his///PRP$///M', 'demise///demise///NN///M']


In [5]:
token_info = tokens[0].split("///")
print(token_info)

['People', 'people', 'NNS', 'B']


In [6]:
parsed_tokens = []
for token in tokens:
    word,lemma,pos,marker = token.split("///")
    parsed_tokens.append({"word": word,"lemma": lemma,"pos": pos,"marker": marker})

parsed_tokens[:5]
    

[{'word': 'People', 'lemma': 'people', 'pos': 'NNS', 'marker': 'B'},
 {'word': 'have', 'lemma': 'have', 'pos': 'VBP', 'marker': 'B'},
 {'word': 'predicted', 'lemma': 'predict', 'pos': 'VBN', 'marker': 'E1'},
 {'word': 'his', 'lemma': 'his', 'pos': 'PRP$', 'marker': 'M'},
 {'word': 'demise', 'lemma': 'demise', 'pos': 'NN', 'marker': 'M'}]

In [7]:

words = []

event1 = None
event2 = None
event1_index = None
event2_index = None

for i,token in enumerate(parsed_tokens):
    words.append(token["word"])

    if marker == "E1":
        event1 = word
        event1_index = i

    if marker == "E2":
        event2 = word
        event2_index = i

text = " ".join(words)

print("Text:",text)
print("Event 1:",event1)
print("Event 1 index:",event1_index)
print("Event 2:",event2)
print("Event 2 index:",event2_index)
print("Relation:", example.attrib["LABEL"])

Text: People have predicted his demise so many times , and the US has tried to hasten it on several occasions .
Event 1: None
Event 1 index: None
Event 2: None
Event 2 index: None
Relation: BEFORE


In [8]:
for item in root:
    if item.attrib["SOURCE"] == "E415":
        print(item.attrib)
        break

{'DOCID': 'ABC19980120.1830.0957', 'SOURCE': 'E415', 'TARGET': 'E417', 'SOURCE_SENTID': '1', 'TARGET_SENTID': '1', 'LABEL': 'BEFORE', 'SENTDIFF': '0'}


In [9]:
for item in root:
    if "speculated///" in item.text and "agreed///" in item.text:
        print(item.attrib)
        print(item.text[:500])
        break

{'DOCID': 'wsj_0938', 'SOURCE': 'E800', 'TARGET': 'E803', 'SOURCE_SENTID': '9', 'TARGET_SENTID': '10', 'LABEL': 'BEFORE', 'SENTDIFF': '1'}
Several///several///JJ///B Cineplex///cineplex///NNP///B analysts///analyst///NNS///B have///have///VBP///B speculated///speculate///VBN///E1 that///that///IN///M outside///outside///JJ///M bids///bid///NNS///M received///receive///VBN///M by///by///IN///M the///the///DT///M committee///committee///NN///M were///be///VBD///M either///either///RB///M disappointingly///disappointingly///RB///M low///low///JJ///M or///or///CC///M for///for///IN///M only///only///JJ///M part///part///NN///M of///of/


In [10]:
tokens = item.text.strip().split()

for i, token in enumerate(tokens):
    word = token.split("///")[0]

    if word == ".":
        print("Period at index:", i)
        print("Tokens around period:", tokens[max(0, i-3):i+5])

Period at index: 23
Tokens around period: ['of///of///IN///M', 'the///the///DT///M', 'company///company///NN///M', './//.///.///M', '"///"///``///M', 'All///all///DT///M', 'this///this///DT///M', 'has///have///VBZ///M']
Period at index: 63
Tokens around period: ['in///in///IN///A', 'Los///los///NNP///A', 'Angeles///angel///NNP///A', './//.///.///A']


In [11]:
for i in range(19, 30):
    parts = tokens[i].split("///")
    print(i, parts[0], parts[-1])

19 part M
20 of M
21 the M
22 company M
23 . M
24 " M
25 All M
26 this M
27 has M
28 really M
29 established M


In [12]:
import pandas as pd
import spacy
from spacy.tokens import Doc
nlp = spacy.load("en_core_web_sm")
data = pd.read_csv("processed_data/train.csv")
text = data.iloc[158]["text"]
words = text.split()
doc = Doc(nlp.vocab, words=words)
for token in doc:
    token.is_sent_start = False
doc[0].is_sent_start = True
doc[24].is_sent_start = True
doc = nlp(doc)
for sent in doc.sents:
    print("Sentence:", sent.text)
    print("Root:", sent.root.text)
    print("Root index:", sent.root.i)
    print("---")

Sentence: Several Cineplex analysts have speculated that outside bids received by the committee were either disappointingly low or for only part of the company .
Root: speculated
Root index: 4
---
Sentence: " All this has really established is that MCA and the Bronfmans have agreed on a price at which they can be bought out , " said Jeffery Logsdon , an analyst with Crowell , Weedon in Los Angeles .
Root: said
Root index: 50
---


In [13]:
for index, row in data.iterrows():
    if row["sent_diff"] != 1 or row["text"] == data.iloc[158]["text"]:
        continue

    words = row["text"].split()
    doc = Doc(nlp.vocab, words=words)
    doc = nlp(doc)

    if len(list(doc.sents)) != 2:
        print("Row:", index)
        print("Event 1:", row["event1"])
        print("Event 2:", row["event2"])
        print("Detected sentences:")
        for sent in doc.sents:
            print("-", sent.text)

        break

Row: 160
Event 1: agreed
Event 2: remains
Detected sentences:
- " All this has really established is that MCA and the Bronfmans have agreed on a price at which they can be bought out , " said Jeffery Logsdon , an analyst with Crowell , Weedon in Los Angeles .
- "
- If a bid materializes at that price , shareholders will have every reason to be glad , but the question of financing still remains . "


In [14]:
words = data.iloc[160]["text"].split()

for i, word in enumerate(words):
    if word == ".":
        print(i, words[max(0, i-3):i+5])

39 ['in', 'Los', 'Angeles', '.', '"', 'If', 'a', 'bid']
65 ['financing', 'still', 'remains', '.', '"']


In [15]:
def find_boundary_candidates(text, event1_index, event2_index):
    words = text.split()
    candidates = []

    for i in range(event1_index + 1, event2_index):
        if words[i] in [".", "?", "!"]:
            next_index = i + 1

            # Look past quotation marks
            while next_index < len(words) and words[next_index] in ['"', "``", "''"]:
                next_index += 1

            if next_index < len(words):
                candidates.append(i + 1)

    return candidates

In [16]:
for index in [158, 160]:
    row = data.iloc[index]

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    print("Row:", index)
    print("Candidates:", candidates)

Row: 158
Candidates: [24]
Row: 160
Candidates: [40]


In [17]:
one_candidate = 0
no_candidate = 0
multiple_candidates = 0

for index, row in data.iterrows():
    if row["sent_diff"] != 1:
        continue

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    if len(candidates) == 1:
        one_candidate += 1
    elif len(candidates) == 0:
        no_candidate += 1
    else:
        multiple_candidates += 1

print("One candidate:", one_candidate)
print("No candidate:", no_candidate)
print("Multiple candidates:", multiple_candidates)

One candidate: 8220
No candidate: 21
Multiple candidates: 14


In [18]:
for index, row in data.iterrows():
    if row["sent_diff"] != 1:
        continue

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    if len(candidates) == 0:
        print("Row:", index)
        print("Text:", row["text"])
        print("Event 1:", row["event1"])
        print("Event 2:", row["event2"])
        break

Row: 5887
Text: The New York Times said in an editorial on Thursday , April 6 : Juan Miguel Gonzalez has made the right decision to travel to the United States on Thursday to affirm his desire to be reunited with his son , Elian .
Event 1: said
Event 2: made


In [19]:
for index, row in data.iterrows():
    if row["sent_diff"] != 1:
        continue

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    if len(candidates) > 1:
        print("Row:", index)
        print("Text:", row["text"])
        print("Event 1:", row["event1"])
        print("Event 2:", row["event2"])
        print("Candidates:", candidates)
        break

Row: 4395
Text: But it has offended some , especially in Arab countries , who now believe that Mr. Bush 's real objectives are the demise of Saddam Hussein and the destruction of the Iraqi military , not just the liberation of Kuwait . " Why have a war ? " asked Abdul Latif Shekar , a customs officer in Egypt , a country participating in the attack on Iraqi troops .
Event 1: offended
Event 2: asked
Candidates: [41, 47]


In [20]:
for index, row in data.iterrows():

    if row["sent_diff"] != 1:
        continue

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    if len(candidates) > 1:
        words = row["text"].split()

        print("Row:", index)

        for candidate in candidates:
            print("Candidate:", candidate)
            print("Context:", " ".join(words[max(0, candidate-5):candidate+5]))

        print("----------------------")

Row: 4395
Candidate: 41
Context: the liberation of Kuwait . " Why have a war
Candidate: 47
Context: Why have a war ? " asked Abdul Latif Shekar
----------------------
Row: 4396
Candidate: 41
Context: the liberation of Kuwait . " Why have a war
Candidate: 47
Context: Why have a war ? " asked Abdul Latif Shekar
----------------------
Row: 4397
Candidate: 41
Context: the liberation of Kuwait . " Why have a war
Candidate: 47
Context: Why have a war ? " asked Abdul Latif Shekar
----------------------
Row: 4398
Candidate: 41
Context: the liberation of Kuwait . " Why have a war
Candidate: 47
Context: Why have a war ? " asked Abdul Latif Shekar
----------------------
Row: 4714
Candidate: 56
Context: living in the workplace ? '' . Justices Sandra Day
Candidate: 58
Context: the workplace ? '' . Justices Sandra Day O'Connor and
----------------------
Row: 4715
Candidate: 56
Context: living in the workplace ? '' . Justices Sandra Day
Candidate: 58
Context: the workplace ? '' . Justices Sandra Day 

In [21]:
for index, row in data.iterrows():

    if row["sent_diff"] != 1:
        continue

    candidates = find_boundary_candidates(
        row["text"],
        int(row["event1_index"]),
        int(row["event2_index"])
    )

    if len(candidates) == 0:
        print("Row:", index)
        print("Event 1:", row["event1"])
        print("Event 2:", row["event2"])
        print("Text:", row["text"])
        print("----------------------")

Row: 5887
Event 1: said
Event 2: made
Text: The New York Times said in an editorial on Thursday , April 6 : Juan Miguel Gonzalez has made the right decision to travel to the United States on Thursday to affirm his desire to be reunited with his son , Elian .
----------------------
Row: 8132
Event 1: lit
Event 2: raised
Text: In Warsaw , Poland , after dark , as fireworks lit the sky , the Polish flag and the flag of NATO were raised side by side near the Tomb of the Unknown Soldier in Pilsudski Square In Hungary , NATO members ' flags were raised outside the Defense Ministry .
----------------------
Row: 8133
Event 1: raised
Event 2: raised
Text: In Warsaw , Poland , after dark , as fireworks lit the sky , the Polish flag and the flag of NATO were raised side by side near the Tomb of the Unknown Soldier in Pilsudski Square In Hungary , NATO members ' flags were raised outside the Defense Ministry .
----------------------
Row: 8339
Event 1: Injured
Event 2: Injured
Text: KENYA : Total D

In [24]:
data = pd.read_csv("processed_data/train.csv")
print(data.head(1))

                  doc_id                                               text  \
0  ABC19980120.1830.0957  People have predicted his demise so many times...   

      event1 event2  event1_index  event2_index relation  label  sent_diff  
0  predicted  tried             2            13   BEFORE      0          0  
